# SPATIAL INTELLIGENCE - COMMUNITY DETECTION & DEGREE CENTRALITY

Detect natural spatial communities in the prison layout, then compute degree centrality at the community scale to understand zone connectivity.

## 1. Import the needed libraries

In [ ]:
from topologicpy.Vertex import Vertex
from topologicpy.Edge import Edge
from topologicpy.Wire import Wire
from topologicpy.Face import Face
from topologicpy.Shell import Shell
from topologicpy.Cell import Cell
from topologicpy.CellComplex import CellComplex
from topologicpy.Cluster import Cluster
from topologicpy.Topology import Topology
from topologicpy.Dictionary import Dictionary
from topologicpy.Helper import Helper
from topologicpy.Grid import Grid
from topologicpy.Graph import Graph
from topologicpy.Color import Color

## 2. Check the TopologicPy version

In [ ]:
print("This notebook requires topologicpy version 0.9.18 or newer.")
print(Helper.Version())

## 3. Configuration

In [ ]:
import os

renderer = "vscode"

# Paths
BASE_DIR = r"E:\IAAC Local GIT Repositories\Graph ML - Environment\Assignment-02_RamonGarcia"
BREP_PATH = os.path.join(BASE_DIR, "geometries", "prison_clean.brep")
ASSETS_DIR = os.path.join(BASE_DIR, "assets")
os.makedirs(ASSETS_DIR, exist_ok=True)

GRID_SIZE = 2

# Image saving - the orchestrator sets this to True
SAVE_IMAGES = True

def save_fig(fig, filename):
    # Save a plotly figure to the assets directory as PNG.
    if not SAVE_IMAGES or fig is None:
        return
    try:
        path = os.path.join(ASSETS_DIR, filename)
        fig.write_image(path, width=1600, height=1200, scale=2)
        print(f"Saved: {path}")
    except Exception as e:
        print(f"Could not save {filename}: {e}")

## 4. Utility functions

In [ ]:
def reset_dictionaries(shell):
    faces = Topology.Faces(shell)
    for i, f in enumerate(faces):
        d = Topology.Dictionary(f)
        keys = Dictionary.Keys(d)
        for key in keys:
            if not key == "face_id":
                d = Dictionary.RemoveKey(d, key)
        f = Topology.SetDictionary(f, d)

def transfer_dicts_by_key(topologies, selectors, key):
    dicts = {}
    for t in topologies:
        d = Topology.Dictionary(t)
        value = Dictionary.ValueAtKey(d, key, None)
        if value:
            dicts[str(value)] = t
    for s in selectors:
        d = Topology.Dictionary(s)
        value = Dictionary.ValueAtKey(d, key, None)
        if value:
            f = dicts.get(str(value), None)
            if f:
                f = Topology.SetDictionary(f, d)

## 5. Load the floor plan and set up the analysis grid

In [ ]:
# Load the cleaned floor plan
floor_plan = Topology.ByBREPPath(BREP_PATH)

# Create a grid overlay
b_r = Wire.BoundingRectangle(floor_plan)
d = Topology.Dictionary(b_r)
xmin = Dictionary.ValueAtKey(d, "xmin")
xmax = Dictionary.ValueAtKey(d, "xmax")
ymin = Dictionary.ValueAtKey(d, "ymin")
ymax = Dictionary.ValueAtKey(d, "ymax")
width = Dictionary.ValueAtKey(d, "width")
length = Dictionary.ValueAtKey(d, "length")
uRange = list(range(0, int(width) + GRID_SIZE, GRID_SIZE))
vRange = list(range(0, int(length) + GRID_SIZE, GRID_SIZE))
grid = Grid.EdgesByDistances(floor_plan, clip=True, uRange=uRange, vRange=vRange)

# Slice the floor plan to create a shell
shell = Topology.Slice(floor_plan, grid)
faces = Topology.Faces(shell)
for i, f in enumerate(faces):
    d = Dictionary.ByKeyValue("face_id", "face_" + str(i + 1))
    f = Topology.SetDictionary(f, d)
print(f"Shell created with {len(faces)} faces")

## 6. Derive the analysis graph

In [ ]:
# Derive the analysis graph from the shell
analysis_graph = Graph.ByTopology(shell)
g_verts = Graph.Vertices(analysis_graph)
print(f"Analysis graph: {len(g_verts)} vertices, {len(Graph.Edges(analysis_graph))} edges")

## 7. Community Detection

Partition the graph into communities - groups of nodes that are more densely connected to each other than to the rest of the network. This reveals the natural spatial zones in the prison layout.

In [ ]:
community_list = Graph.CommunityPartition(analysis_graph, colorScale="thermal")

Transfer results to the shell faces

In [ ]:
reset_dictionaries(shell)
_ = transfer_dicts_by_key(faces, g_verts, "face_id")

## 8. Show community partition

In [ ]:
fig = Topology.Show(faces,
              faceColorKey="cp_color",
              faceOpacity=1,
              showEdges=False,
              showVertices=False,
              camera=[0, 0, 6],
              backgroundColor="black",
              width=800, height=600,
              showFigure=False, renderer=renderer)
save_fig(fig, "09_community_partition.png")

## 9. Build community-level zones

Group the faces by community, extract each group's outer boundary, and create simplified zone faces.

In [ ]:
bins = Topology.BinByDictionaryKey(faces, key="community")
bin_dict = bins[0]
keys = list(bin_dict.keys())
face_groups = []
for key in keys:
    bin_faces = bin_dict[key]
    temp_shell = Shell.ByFaces(bin_faces)
    eb = Shell.ExternalBoundary(temp_shell)
    eb = Wire.RemoveCollinearEdges(eb)
    eb = Face.ByWire(eb)
    face_groups.append(eb)

print(f"Detected {len(face_groups)} communities")

## 10. Show community zone boundaries

In [ ]:
fig = Topology.Show(face_groups,
              faceOpacity=1,
              showEdges=True,
              edgeWidth=8,
              edgeColor="grey",
              camera=[0, 0, 6],
              backgroundColor="black",
              width=800, height=600,
              showFigure=False, renderer=renderer)
save_fig(fig, "10_community_zones.png")

## 11. Create a graph from the community zones

In [ ]:
new_shell = Shell.ByFaces(face_groups)
new_graph = Graph.ByTopology(new_shell)
new_verts = Graph.Vertices(new_graph)

# Style the community graph nodes
for v in new_verts:
    d = Dictionary.ByKeysValues(["color", "size"], ["red", 12])
    v = Topology.SetDictionary(v, d)

## 12. Show the community graph overlaid on the shell

In [ ]:
fig = Topology.Show(new_shell, new_graph,
              faceOpacity=0.9,
              showEdges=True,
              showVertices=True,
              vertexSizeKey="size",
              vertexColorKey="color",
              camera=[0, 0, 6],
              backgroundColor="black",
              width=800, height=600,
              showFigure=False, renderer=renderer)
save_fig(fig, "11_community_graph.png")

## 13. Compute degree centrality at community scale

In [ ]:
degree_centralities = Graph.DegreeCentrality(new_graph, normalize=False)

## 14. Interpolate degree centrality back to the original grid

Transfer the community-scale degree values back to the fine grid using spatial interpolation.

In [ ]:
for v in g_verts:
    new_v = Vertex.InterpolateValue(v, vertices=new_verts, n=3, key="degree_centrality")

minValue = min(degree_centralities)
maxValue = max(degree_centralities)
for v in g_verts:
    d = Topology.Dictionary(v)
    d_c = Dictionary.ValueAtKey(d, "degree_centrality")
    color = Color.AnyToHex(Color.ByValueInRange(d_c, minValue=minValue, maxValue=maxValue, colorScale="thermal"))
    d = Dictionary.SetValueAtKey(d, "dc_color", color)
    v = Topology.SetDictionary(v, d)

Transfer to the original shell faces

In [ ]:
reset_dictionaries(shell)
faces = Topology.Faces(shell)
_ = transfer_dicts_by_key(faces, g_verts, "face_id")

## 15. Show degree centrality heatmap

In [ ]:
fig = Topology.Show(faces,
              faceColorKey="dc_color",
              faceOpacity=1,
              showEdges=False,
              showVertices=False,
              camera=[0, 0, 6],
              backgroundColor="black",
              width=800, height=600,
              showFigure=False, renderer=renderer)
save_fig(fig, "12_degree_centrality.png")